In [2]:
import random
import numpy as np
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [3]:
# =====================================================
# QUESTION 3 — TEXT SUMMARIZATION
# CNN/DailyMail: TextRank vs BART
# Metrics: ROUGE, BLEU, METEOR, BERTScore
# =====================================================

# 1. Install libraries
!pip install -q "datasets<4.0.0" transformers evaluate rouge_score bert_score nltk networkx scikit-learn sacrebleu

# =====================================================
# 2. Imports and seed
# =====================================================

import random
import numpy as np
import torch
import pandas as pd
import nltk
import re
import networkx as nx

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import evaluate

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

# NLTK resources
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("omw-1.4")

# =====================================================
# 3. Load CNN/DailyMail dataset
# =====================================================

dataset = load_dataset("cnn_dailymail", "3.0.0")

print("\nDataset structure:")
print(dataset)

print("\nDataset keys:")
print(dataset["validation"][0].keys())

# Use a small subset for speed
NUM_EXAMPLES = 20

subset = dataset["validation"].shuffle(seed=SEED).select(range(NUM_EXAMPLES))

articles = subset["article"]
references = subset["highlights"]

print("\nNumber of examples:", len(articles))

print("\nExample article:")
print(articles[0][:1000])

print("\nReference summary:")
print(references[0])

# =====================================================
# 4. TextRank Extractive Summarization
# =====================================================

def clean_sentence(sentence):
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence.strip()


def textrank_summarize(article, top_n=3):
    """
    Simple TextRank implementation:
    1. Split article into sentences.
    2. Represent sentences with TF-IDF.
    3. Compute cosine similarity between sentences.
    4. Apply PageRank on the similarity graph.
    5. Select top-ranked sentences as extractive summary.
    """

    sentences = nltk.sent_tokenize(article)
    sentences = [clean_sentence(s) for s in sentences if len(clean_sentence(s)) > 20]

    if len(sentences) == 0:
        return ""

    if len(sentences) <= top_n:
        return " ".join(sentences)

    vectorizer = TfidfVectorizer(stop_words="english")
    sentence_vectors = vectorizer.fit_transform(sentences)

    similarity_matrix = cosine_similarity(sentence_vectors)

    graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(graph)

    ranked_sentences = sorted(
        ((scores[i], i, sentence) for i, sentence in enumerate(sentences)),
        reverse=True
    )

    selected = sorted(ranked_sentences[:top_n], key=lambda x: x[1])

    summary = " ".join([sentence for _, _, sentence in selected])

    return summary


print("\nTesting TextRank on one example:")
print(textrank_summarize(articles[0], top_n=3))

textrank_summaries = []

print("\nGenerating TextRank summaries...")

for i, article in enumerate(articles):
    print(f"TextRank summarizing example {i+1}/{len(articles)}")
    summary = textrank_summarize(article, top_n=3)
    textrank_summaries.append(summary)

print("\nGenerated TextRank summaries:", len(textrank_summaries))

# =====================================================
# 5. BART Abstractive Summarization
# =====================================================

bart_model_name = "facebook/bart-large-cnn"

print("\nLoading BART model...")

bart_tokenizer = AutoTokenizer.from_pretrained(bart_model_name)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(bart_model_name).to(device)

def bart_summarize(article):
    """
    Generate abstractive summary using BART.
    CNN/DailyMail articles can be long, so input is truncated to 1024 tokens.
    """

    inputs = bart_tokenizer(
        article,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    summary_ids = bart_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=130,
        min_length=30,
        num_beams=4,
        early_stopping=True
    )

    summary = bart_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary


bart_summaries = []

print("\nGenerating BART summaries...")

for i, article in enumerate(articles):
    print(f"BART summarizing example {i+1}/{len(articles)}")
    summary = bart_summarize(article)
    bart_summaries.append(summary)

print("\nGenerated BART summaries:", len(bart_summaries))

print("\nExample BART summary:")
print(bart_summaries[0])

# =====================================================
# 6. Evaluation Metrics
# =====================================================

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")


def evaluate_summaries(predictions, references, method_name):
    print(f"\nEvaluating {method_name}...")

    # ROUGE
    rouge_results = rouge_metric.compute(
        predictions=predictions,
        references=references
    )

    # BLEU
    # evaluate BLEU expects references as list of lists
    bleu_results = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )

    # METEOR
    meteor_results = meteor_metric.compute(
        predictions=predictions,
        references=references
    )

    # BERTScore
    bertscore_results = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="en"
    )

    avg_bertscore_f1 = float(np.mean(bertscore_results["f1"]))

    results = {
        "Method": method_name,
        "ROUGE-1": rouge_results["rouge1"],
        "ROUGE-2": rouge_results["rouge2"],
        "ROUGE-L": rouge_results["rougeL"],
        "BLEU": bleu_results["bleu"],
        "METEOR": meteor_results["meteor"],
        "BERTScore-F1": avg_bertscore_f1
    }

    return results


textrank_results = evaluate_summaries(
    textrank_summaries,
    references,
    "TextRank"
)

bart_results = evaluate_summaries(
    bart_summaries,
    references,
    "BART"
)

results_df_q3 = pd.DataFrame([textrank_results, bart_results])

print("\n=========================")
print("FINAL Q3 RESULTS TABLE")
print("=========================")
print(results_df_q3)

# =====================================================
# 7. Qualitative Examples
# =====================================================

print("\n=========================")
print("QUALITATIVE EXAMPLES")
print("=========================")

for i in range(3):
    print("=" * 100)
    print(f"Example {i+1}")
    print("=" * 100)

    print("\nARTICLE:")
    print(articles[i][:1500], "...")

    print("\nREFERENCE SUMMARY:")
    print(references[i])

    print("\nTEXTRANK SUMMARY:")
    print(textrank_summaries[i])

    print("\nBART SUMMARY:")
    print(bart_summaries[i])

# =====================================================
# 8. Save outputs for report
# =====================================================

qualitative_data = []

for i in range(len(articles)):
    qualitative_data.append({
        "article": articles[i],
        "reference_summary": references[i],
        "textrank_summary": textrank_summaries[i],
        "bart_summary": bart_summaries[i]
    })

qualitative_df = pd.DataFrame(qualitative_data)

results_df_q3.to_csv("q3_summarization_results.csv", index=False)
qualitative_df.to_csv("q3_qualitative_examples.csv", index=False)

print("\nSaved files:")
print("q3_summarization_results.csv")
print("q3_qualitative_examples.csv")

Device: cuda


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

Dataset keys:
dict_keys(['article', 'highlights', 'id'])

Number of examples: 20

Example article:
Jarryd Hayne's move to the NFL is a boost for rugby league in the United States, it has been claimed. The Australia international full-back or centre quit the National Rugby League in October to try his luck in American football and was this week given a three-year contract with the San Francisco 49ers. Peter Illfield, chairman of US Association of Rugby League, said: 'Jarryd, at 27, is one of the most gifted and talented rugby league players in Australia. He is an extraordinary athlete. Jarryd Hayne (right) has signed with the San Francisco 4

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]


Generating BART summaries...
BART summarizing example 1/20
BART summarizing example 2/20
BART summarizing example 3/20
BART summarizing example 4/20
BART summarizing example 5/20
BART summarizing example 6/20
BART summarizing example 7/20
BART summarizing example 8/20
BART summarizing example 9/20
BART summarizing example 10/20
BART summarizing example 11/20
BART summarizing example 12/20
BART summarizing example 13/20
BART summarizing example 14/20
BART summarizing example 15/20
BART summarizing example 16/20
BART summarizing example 17/20
BART summarizing example 18/20
BART summarizing example 19/20
BART summarizing example 20/20

Generated BART summaries: 20

Example BART summary:
Jarryd Hayne quit the NRL in October to try his luck in American football. The 27-year-old has signed a three-year contract with the San Francisco 49ers. Hayne could play at full back or centre in rugby league. He is expected to be a running back for the 49ers .


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



Evaluating TextRank...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Evaluating BART...

FINAL Q3 RESULTS TABLE
     Method   ROUGE-1   ROUGE-2   ROUGE-L      BLEU    METEOR  BERTScore-F1
0  TextRank  0.350987  0.139082  0.233053  0.096463  0.322928      0.866432
1      BART  0.397239  0.172273  0.281219  0.162720  0.336573      0.875359

QUALITATIVE EXAMPLES
Example 1

ARTICLE:
Jarryd Hayne's move to the NFL is a boost for rugby league in the United States, it has been claimed. The Australia international full-back or centre quit the National Rugby League in October to try his luck in American football and was this week given a three-year contract with the San Francisco 49ers. Peter Illfield, chairman of US Association of Rugby League, said: 'Jarryd, at 27, is one of the most gifted and talented rugby league players in Australia. He is an extraordinary athlete. Jarryd Hayne (right) has signed with the San Francisco 49ers after quitting the NRL in October . Hayne, who played rugby league for Australia, has signed a three year contract with the 49ers . 